# Testing for the 5D Integration

In [1]:
import numpy as np
import dadi 
from matplotlib import pyplot as plt
import dadi.Polyploidy.Integration as PolyInt
from dadi.Polyploidy import wrightfisher as WF
import time

## Tests against dadi

In [2]:
# plotting function
# this is useful for debugging, but it is much easier to just use
# assert np.allclose(phi_poly, phi_dadi)

def phi_5D_plot(phi_poly, phi_dadi, edges, axis1, axis2, axis3, xlab, ylab, plotall=False):
    # marginalize once
    phi_dadi1 = np.sum(phi_dadi, axis=axis1)
    phi_poly1 = np.sum(phi_poly, axis=axis1)
    # and then twice
    phi_dadi1 = np.sum(phi_dadi1, axis=axis2)
    phi_poly1 = np.sum(phi_poly1, axis=axis2)
    # and then a third time
    phi_dadi1 = np.sum(phi_dadi1, axis=axis3)
    phi_poly1 = np.sum(phi_poly1, axis=axis3)

    if plotall:
        plt.pcolormesh(edges, edges, phi_poly1.T, cmap='viridis', shading='auto')
        plt.colorbar(label='Density')
        plt.xlabel(xlab)
        plt.ylabel(ylab)
        plt.title('PolyInt')
        plt.show()

        plt.pcolormesh(edges, edges, phi_dadi1.T, cmap='viridis', shading='auto')
        plt.colorbar(label='Density')
        plt.xlabel(xlab)
        plt.ylabel(ylab)
        plt.title('dadi')
        plt.show()

        plt.pcolormesh(edges, edges, phi_poly1.T - phi_dadi1.T, cmap='viridis', shading='auto')
        plt.colorbar(label='Density')
        plt.xlabel(xlab)
        plt.ylabel(ylab)
        plt.title('PolyInt - dadi')
        plt.show()

    plt.pcolormesh(edges, edges, np.abs(phi_poly1.T - phi_dadi1.T)/np.abs(phi_dadi1.T), cmap='viridis', shading='auto')
    plt.colorbar(label='Density')
    plt.xlabel(xlab)
    plt.ylabel(ylab)
    plt.title('(PolyInt - dadi)/dadi')
    plt.show()
    

### Test against all diploids

In [5]:
xx = dadi.Numerics.default_grid(pts=13)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)
phi = dadi.PhiManip.phi_4D_to_5D(phi, 1/4, 1/4, 1/4, xx, xx, xx, xx, xx)

dipflag = PolyInt.PloidyType.DIPLOID
T = .05

gamma1 = -1
gamma2 = -4
gamma3 = -2
gamma4 = 0
gamma5 = -3

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5
h5 = 0.5

m12 = .1
m13 = 0
m14 = 1
m15 = 0.2
m21 = .01
m23 = .15
m24 = 0.4
m25 = 0.3
m31 = .2
m32 = .05
m34 = 0
m35 = 0.7
m41 = 0.3
m42 = 0
m43 = 0.2
m45 = 0
m51 = .2
m52 = .1
m53 = 1
m54 = .01

sel_dip1 = {'gamma': gamma1, 'h': h1}
sel_dip2 = {'gamma': gamma2, 'h': h2}
sel_dip3 = {'gamma': gamma3, 'h': h3}
sel_dip4 = {'gamma': gamma4, 'h': h4}
sel_dip5 = {'gamma': gamma5, 'h': h5}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu3 = 1.2
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu4 = 1.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu5 = 2.2
nu5_func = lambda t: np.exp(np.log(nu5)*t/T)

phi_poly = PolyInt.five_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=dipflag, ploidyflag2=dipflag, ploidyflag3=dipflag, ploidyflag4=dipflag, ploidyflag5=dipflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func,
                              m12=m12, m13=m13, m14=m14, m15=m15, 
                              m21=m21, m23=m23, m24=m24, m25=m25,
                              m31=m31, m32=m32, m34=m34, m35=m35,
                              m41=m41, m42=m42, m43=m43, m45=m45,
                              m51=m51, m52=m52, m53=m53, m54=m54, theta0=1)

phi_dadi = dadi.Integration.five_pops(phi.copy(), xx, T=T, theta0=1, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4, gamma5=gamma5,
                                    h1=h1, h2=h2, h3=h3, h4=h4, h5=h5,
                                    m12=m12, m13=m13, m14=m14, m15=m15, 
                                    m21=m21, m23=m23, m24=m24, m25=m25,
                                    m31=m31, m32=m32, m34=m34, m35=m35,
                                    m41=m41, m42=m42, m43=m43, m45=m45,
                                    m51=m51, m52=m52, m53=m53, m54=m54,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# # note, we don't plot every combination, since that would be 4! = 24 plots
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 0, 'Pop 4', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 1, 'Pop 3', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 2, 'Pop 3', 'Pop 4')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 2, 'Pop 1', 'Pop 2')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 1, 'Pop 1', 'Pop 5')

### Test against all autos with rescaled params and no mutation

In [ ]:
xx = dadi.Numerics.default_grid(pts=13)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)
phi = dadi.PhiManip.phi_4D_to_5D(phi, 1/4, 1/4, 1/4, xx, xx, xx, xx, xx)

autoflag = PolyInt.PloidyType.AUTO
T = .1

gamma1 = -1
gamma2 = -4
gamma3 = -2
gamma4 = 0
gamma5 = -3

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5
h5 = 0.5

m12 = .1
m13 = 0
m14 = 1
m15 = 0.2
m21 = .01
m23 = .15
m24 = 0.4
m25 = 0.3
m31 = .2
m32 = .05
m34 = 0
m35 = 0.7
m41 = 0.3
m42 = 0
m43 = 0.2
m45 = 0
m51 = .2
m52 = .1
m53 = 1
m54 = .01

sel_dip1 = {'gamma': gamma1}
sel_dip2 = {'gamma': gamma2}
sel_dip3 = {'gamma': gamma3}
sel_dip4 = {'gamma': gamma4}
sel_dip5 = {'gamma': gamma5}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu1_func_auto = lambda t: np.exp(np.log(nu1)*t/(2*T))
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu2_func_auto = lambda t: np.exp(np.log(nu2)*t/(2*T))
nu3 = 1.2
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu3_func_auto = lambda t: np.exp(np.log(nu3)*t/(2*T))
nu4 = 1.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu4_func_auto = lambda t: np.exp(np.log(nu4)*t/(2*T))
nu5 = 2.2
nu5_func = lambda t: np.exp(np.log(nu5)*t/T)
nu5_func_auto = lambda t: np.exp(np.log(nu5)*t/(2*T))

phi_poly = PolyInt.five_pops(phi.copy(), xx, T=2*T, 
                              ploidyflag1=autoflag, ploidyflag2=autoflag, ploidyflag3=autoflag, ploidyflag4=autoflag, ploidyflag5=autoflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func_auto, nu2=nu2_func_auto, nu3=nu3_func_auto, nu4=nu4_func_auto, nu5=nu5_func_auto,    
                              m12=m12/2, m13=m13/2, m14=m14/2, m15=m15/2, 
                              m21=m21/2, m23=m23/2, m24=m24/2, m25=m25/2,
                              m31=m31/2, m32=m32/2, m34=m34/2, m35=m35/2,
                              m41=m41/2, m42=m42/2, m43=m43/2, m45=m45/2,
                              m51=m51/2, m52=m52/2, m53=m53/2, m54=m54/2, theta0=0)

phi_dadi = dadi.Integration.five_pops(phi.copy(), xx, T=T, theta0=0, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4, gamma5=gamma5,
                                    h1=h1, h2=h2, h3=h3, h4=h4, h5=h5,
                                    m12=m12, m13=m13, m14=m14, m15=m15, 
                                    m21=m21, m23=m23, m24=m24, m25=m25,
                                    m31=m31, m32=m32, m34=m34, m35=m35,
                                    m41=m41, m42=m42, m43=m43, m45=m45,
                                    m51=m51, m52=m52, m53=m53, m54=m54,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 0, 'Pop 4', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 1, 'Pop 3', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 2, 'Pop 3', 'Pop 4')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 2, 'Pop 1', 'Pop 2')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 1, 'Pop 1', 'Pop 5')

### Test against a mix of allotetraploids and diploids

In [10]:
xx = dadi.Numerics.default_grid(pts=13)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)
phi = dadi.PhiManip.phi_4D_to_5D(phi, 1/4, 1/4, 1/4, xx, xx, xx, xx, xx)

dipflag = PolyInt.PloidyType.DIPLOID
alloaflag = PolyInt.PloidyType.ALLOa
allobflag = PolyInt.PloidyType.ALLOb
T = .1

gamma1 = 0
gamma2 = 0
gamma3 = -2
gamma4 = 0
gamma5 = 0

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5
h5 = 0.5

m13 = 0
m14 = 1
m15 = 0.2
m23 = .15
m24 = 0.4
m25 = 0.3
m31 = .2
m32 = .05
m34 = 0
m35 = 0.7
m41 = 0.3
m42 = 0
m43 = 0.2
m51 = .2
m52 = .1
m53 = 1

# following pairs specify a single exchange parameter
m12 = m21 = .1
m54 = m45 = .01

sel_dip1 = {'gamma': gamma1}
sel_dip2 = {'gamma': gamma2}
sel_dip3 = {'gamma': gamma3}
sel_dip4 = {'gamma': gamma4}
sel_dip5 = {'gamma': gamma5}

nu1 = 0.8
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 0.8
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu3 = 1.5
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu4 = 2.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu5 = 2.2
nu5_func = lambda t: np.exp(np.log(nu5)*t/T)

phi_poly = PolyInt.five_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=alloaflag, ploidyflag2=allobflag, ploidyflag3=dipflag, ploidyflag4=alloaflag, ploidyflag5=allobflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func,    
                              m12=m12, m13=m13, m14=m14, m15=m15, 
                              m21=m21, m23=m23, m24=m24, m25=m25,
                              m31=m31, m32=m32, m34=m34, m35=m35,
                              m41=m41, m42=m42, m43=m43, m45=m45,
                              m51=m51, m52=m52, m53=m53, m54=m54, theta0=1)

phi_dadi = dadi.Integration.five_pops(phi.copy(), xx, T=T, theta0=1, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4, gamma5=gamma5,
                                    h1=h1, h2=h2, h3=h3, h4=h4, h5=h5,
                                    m12=m12, m13=m13, m14=m14, m15=m15, 
                                    m21=m21, m23=m23, m24=m24, m25=m25,
                                    m31=m31, m32=m32, m34=m34, m35=m35,
                                    m41=m41, m42=m42, m43=m43, m45=m45,
                                    m51=m51, m52=m52, m53=m53, m54=m54,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 0, 'Pop 4', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 1, 'Pop 3', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 2, 'Pop 3', 'Pop 4')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 2, 'Pop 1', 'Pop 2')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 1, 'Pop 1', 'Pop 5')

### Test against all autotetraploids with rescaled params and no mutation

In [9]:
xx = dadi.Numerics.default_grid(pts=15)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)
phi = dadi.PhiManip.phi_4D_to_5D(phi, 1/4, 1/4, 1/4, xx, xx, xx, xx, xx)

autoflag = PolyInt.PloidyType.AUTOHEX
T = .05

gamma1 = -1
gamma2 = -4
gamma3 = -2
gamma4 = 0
gamma5 = -3

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5
h5 = 0.5

m12 = .1
m13 = 0
m14 = 1
m15 = 0.2
m21 = .01
m23 = .15
m24 = 0.4
m25 = 0.3
m31 = .2
m32 = .05
m34 = 0
m35 = 0.7
m41 = 0.3
m42 = 0
m43 = 0.2
m45 = 0
m51 = .2
m52 = .1
m53 = 1
m54 = .01

sel_dip1 = {'gamma': gamma1}
sel_dip2 = {'gamma': gamma2}
sel_dip3 = {'gamma': gamma3}
sel_dip4 = {'gamma': gamma4}
sel_dip5 = {'gamma': gamma5}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu1_func_auto = lambda t: np.exp(np.log(nu1)*t/(3*T))
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu2_func_auto = lambda t: np.exp(np.log(nu2)*t/(3*T))
nu3 = 1.2
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu3_func_auto = lambda t: np.exp(np.log(nu3)*t/(3*T))
nu4 = 1.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu4_func_auto = lambda t: np.exp(np.log(nu4)*t/(3*T))
nu5 = 2.2
nu5_func = lambda t: np.exp(np.log(nu5)*t/T)
nu5_func_auto = lambda t: np.exp(np.log(nu5)*t/(3*T))

phi_poly = PolyInt.five_pops(phi.copy(), xx, T=3*T, 
                              ploidyflag1=autoflag, ploidyflag2=autoflag, ploidyflag3=autoflag, ploidyflag4=autoflag, ploidyflag5=autoflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func_auto, nu2=nu2_func_auto, nu3=nu3_func_auto, nu4=nu4_func_auto, nu5=nu5_func_auto,    
                              m12=m12/3, m13=m13/3, m14=m14/3, m15=m15/3, 
                              m21=m21/3, m23=m23/3, m24=m24/3, m25=m25/3,
                              m31=m31/3, m32=m32/3, m34=m34/3, m35=m35/3,
                              m41=m41/3, m42=m42/3, m43=m43/3, m45=m45/3,
                              m51=m51/3, m52=m52/3, m53=m53/3, m54=m54/3, theta0=0)

phi_dadi = dadi.Integration.five_pops(phi.copy(), xx, T=T, theta0=0, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4, gamma5=gamma5,
                                    h1=h1, h2=h2, h3=h3, h4=h4, h5=h5,
                                    m12=m12, m13=m13, m14=m14, m15=m15, 
                                    m21=m21, m23=m23, m24=m24, m25=m25,
                                    m31=m31, m32=m32, m34=m34, m35=m35,
                                    m41=m41, m42=m42, m43=m43, m45=m45,
                                    m51=m51, m52=m52, m53=m53, m54=m54,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 0, 'Pop 4', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 1, 'Pop 3', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 2, 'Pop 3', 'Pop 4')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 2, 'Pop 1', 'Pop 2')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 1, 'Pop 1', 'Pop 5')

### Finally, test a mix of diploids and 2+2+2 hexaploids without selection in the hexaploid subgenomes

In [12]:
xx = dadi.Numerics.default_grid(pts=13)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)
phi = dadi.PhiManip.phi_4D_to_5D(phi, 1/4, 1/4, 1/4, xx, xx, xx, xx, xx)

dipflag = PolyInt.PloidyType.DIPLOID
hexaflag = PolyInt.PloidyType.HEXa
hexbflag = PolyInt.PloidyType.HEXb
hexcflag = PolyInt.PloidyType.HEXc  

T = .1

# we can have some selection parameters for the diploids
gamma1 = -1
gamma2 = 1
gamma3 = 0
gamma4 = 0
gamma5 = 0

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5
h5 = 0.5

m12 = .1
m13 = 0
m14 = 1
m15 = 0.2
m21 = .1
m23 = .15
m24 = 0.4
m25 = 0.3
m31 = .2
m32 = .05
m41 = 0.3
m42 = 0
m51 = .2
m52 = .1

# following pairs specify a single exchange parameter
m34 = m43 = .2
m45 = m54 = .01
m35 = m53 = 0.7

sel_dip1 = {'gamma': gamma1}
sel_dip2 = {'gamma': gamma2}
sel_dip3 = {'gamma': gamma3}
sel_dip4 = {'gamma': gamma4}
sel_dip5 = {'gamma': gamma5}

nu1 = 1.3
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 0.8
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu3 = 2.2
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu4 = 2.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu5 = 2.2
nu5_func = lambda t: np.exp(np.log(nu5)*t/T)

phi_poly = PolyInt.five_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=dipflag, ploidyflag2=dipflag, ploidyflag3=hexaflag, ploidyflag4=hexbflag, ploidyflag5=hexcflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func,    
                              m12=m12, m13=m13, m14=m14, m15=m15, 
                              m21=m21, m23=m23, m24=m24, m25=m25,
                              m31=m31, m32=m32, m34=m34, m35=m35,
                              m41=m41, m42=m42, m43=m43, m45=m45,
                              m51=m51, m52=m52, m53=m53, m54=m54, theta0=1)

phi_dadi = dadi.Integration.five_pops(phi.copy(), xx, T=T, theta0=1, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4, gamma5=gamma5,
                                    h1=h1, h2=h2, h3=h3, h4=h4, h5=h5,
                                    m12=m12, m13=m13, m14=m14, m15=m15, 
                                    m21=m21, m23=m23, m24=m24, m25=m25,
                                    m31=m31, m32=m32, m34=m34, m35=m35,
                                    m41=m41, m42=m42, m43=m43, m45=m45,
                                    m51=m51, m52=m52, m53=m53, m54=m54,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 0, 'Pop 4', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 1, 'Pop 3', 'Pop 5')
# phi_5D_plot(phi_poly, phi_dadi, edges, 0, 0, 2, 'Pop 3', 'Pop 4')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 2, 'Pop 1', 'Pop 2')
# phi_5D_plot(phi_poly, phi_dadi, edges, 2, 2, 1, 'Pop 1', 'Pop 5')

## Tests for GPU Code (not against dadi)

In [ ]:
# Test Alloautohexaploids (4+2 hexaploids)
xx = dadi.Numerics.default_grid(pts=13)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)
phi = dadi.PhiManip.phi_4D_to_5D(phi, 1/4, 1/4, 1/4, xx, xx, xx, xx, xx)

dipflag = PolyInt.PloidyType.DIPLOID
hex_tetraflag = PolyInt.PloidyType.HEX_tetra
hex_dipflag = PolyInt.PloidyType.HEX_dip
T = .2

gamma1 = -2
gamma2 = -2
gamma3 = -2
gamma4 = 1
gamma5 = 1

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5
h5 = 0.5

m13 = 0
m14 = 1
m15 = 0.2
m23 = .15
m24 = 0.4
m25 = 0.3
m31 = .2
m32 = .05
m34 = 0
m35 = 0.7
m41 = 0.3
m42 = 0
m43 = 0.2
m51 = .2
m52 = .1
m53 = 1

# following pairs specify a single exchange parameter
m12 = m21 = 2
m54 = m45 = .01

sel_dip1 = {'gamma': gamma1}
sel_dip2 = {'gamma': gamma2}
sel_dip3 = {'gamma': gamma3}
sel_dip4 = {'gamma': gamma4}
sel_dip5 = {'gamma': gamma5}

nu1 = 0.8
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 0.8
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu3 = 1.5
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu4 = 2.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu5 = 2.2
nu5_func = lambda t: np.exp(np.log(nu5)*t/T)

phi_cpu = PolyInt.five_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=hex_tetraflag, ploidyflag2=hex_dipflag, ploidyflag3=dipflag, ploidyflag4=hex_tetraflag, ploidyflag5=hex_dipflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func,    
                              m12=m12, m13=m13, m14=m14, m15=m15, 
                              m21=m21, m23=m23, m24=m24, m25=m25,
                              m31=m31, m32=m32, m34=m34, m35=m35,
                              m41=m41, m42=m42, m43=m43, m45=m45,
                              m51=m51, m52=m52, m53=m53, m54=m54, theta0=1)

PolyInt.cuda_enabled = True

phi_gpu = PolyInt.five_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=hex_tetraflag, ploidyflag2=hex_dipflag, ploidyflag3=dipflag, ploidyflag4=hex_tetraflag, ploidyflag5=hex_dipflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4, sel_dict5=sel_dip5,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func, nu5=nu5_func,    
                              m12=m12, m13=m13, m14=m14, m15=m15, 
                              m21=m21, m23=m23, m24=m24, m25=m25,
                              m31=m31, m32=m32, m34=m34, m35=m35,
                              m41=m41, m42=m42, m43=m43, m45=m45,
                              m51=m51, m52=m52, m53=m53, m54=m54, theta0=1)